## STEP1 진단
정제하기 전에 **무엇이 얼마나 망가졌는지** 파악합니다.

세 파일을 읽고 아래를 확인해 표로 정리하세요.

In [1]:
import pandas as pd
import numpy as np
import unicodedata

from config import path, ENCODING

stations = pd.read_csv(path('stations.csv'), encoding=ENCODING, keep_default_na=False)
raw_bikes = pd.read_csv(path('raw-bikes.csv'), encoding=ENCODING, keep_default_na=False)
raw_rentals = pd.read_csv(path('raw-rentals.csv'), encoding=ENCODING, keep_default_na=False)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)


In [2]:
stations.info()


<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   station_id    30 non-null     str  
 1   station_name  30 non-null     str  
 2   district      30 non-null     str  
dtypes: str(3)
memory usage: 852.0 bytes


In [3]:
raw_bikes.info()

<class 'pandas.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   bike_id           55 non-null     str  
 1   station_id        55 non-null     str  
 2   bike_type         55 non-null     str  
 3   gear_count        55 non-null     str  
 4   manufacture_year  55 non-null     str  
 5   daily_fee         55 non-null     str  
dtypes: str(6)
memory usage: 2.7 KB


In [4]:
raw_rentals.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   rental_id       15000 non-null  str  
 1   bike_id         15000 non-null  str  
 2   user_id         15000 non-null  str  
 3   rent_time       15000 non-null  str  
 4   return_time     15000 non-null  str  
 5   distance_km     15000 non-null  str  
 6   fee             15000 non-null  str  
 7   payment_method  15000 non-null  str  
dtypes: str(8)
memory usage: 937.6 KB


In [5]:
print(raw_bikes['bike_id'].nunique())   # 50행

50


## STEP 2 · 자전거 마스터 정제

`raw-bikes.csv` 를 정제합니다.

In [6]:
# station_id — 대문자로 통일, 빈 값은 결측으로 처리
raw_bikes['station_id'] = raw_bikes['station_id'].str.upper().replace(r'^\s*$', np.nan, regex=True)

print(raw_bikes['station_id'].unique())

<StringArray>
['S13', 'S25', 'S19', 'S11', 'S26', 'S05', 'S23', 'S20', 'S03', 'S22', 'S10',
 'S08', 'S12', 'S18', 'S30', 'S15', 'S24', 'S28', 'S21',   nan, 'S06', 'S14',
 'S02', 'S04', 'S27', 'S01']
Length: 26, dtype: str


In [7]:
# bike_type — 일반 / 전동 두 가지로 통일
print(raw_bikes['bike_type'].unique())
raw_bikes['bike_type'] = raw_bikes['bike_type'].apply(
    lambda x: unicodedata.normalize("NFKC", str(x)) if pd.notnull(x) else x
)

raw_bikes['bike_type'] = (
    raw_bikes['bike_type']
    .str.replace(' ', '')
    .replace({
        '일반형': '일반',
        'electric': '전동'
    })
)
print(raw_bikes['bike_type'].unique())

<StringArray>
['일반　', '일 반', '일반', '전동', 'electric', '일반형']
Length: 6, dtype: str
<StringArray>
['일반', '전동']
Length: 2, dtype: str


In [8]:
# gear_count — 단위 문자를 제거하고 정수로
raw_bikes['gear_count'] = pd.to_numeric(raw_bikes['gear_count'].astype(str).str.replace('단', ''), errors='coerce')

print(raw_bikes['gear_count'].unique())

[nan  3.  7.]


In [9]:
# daily_fee — 콤마 제거 후 정수로
raw_bikes['daily_fee'] = pd.to_numeric(raw_bikes['daily_fee'].astype(str).str.replace(',', ''), errors='coerce')

print(raw_bikes['daily_fee'].unique())

[1000 2000]


In [10]:
# manufacture_year — 정수로 (변환 실패는 결측으로)
raw_bikes['manufacture_year'] = pd.to_numeric(raw_bikes['manufacture_year'], errors='coerce')

print(raw_bikes['manufacture_year'].unique())

[2019. 2017. 2018.   nan 2015. 2016. 2020. 2023. 2022. 2021.]


In [11]:
# 중복 제거 — bike_id 기준
raw_bikes = raw_bikes.drop_duplicates(subset='bike_id', keep='first').reset_index(drop=True)

print(raw_bikes['bike_id'].duplicated().sum())



0


## STEP 3 · 대여 기록 정제

`raw-rentals.csv` 를 정제합니다.

In [12]:
# distance_km, fee — 콤마 제거 후 숫자로. 변환 실패는 결측으로
cols_to_numeric = ['distance_km', 'fee']

for col in cols_to_numeric:
    raw_rentals[col] = pd.to_numeric(
        raw_rentals[col].astype(str).str.replace(',', ''), 
        errors='coerce'
    )
    print(raw_rentals[col].dtype)

float64
int64


In [13]:

# rent_time, return_time — datetime 으로 통일 (형식 3종 혼재)
cols_to_datetime = ['rent_time', 'return_time']
for col in cols_to_datetime:
    raw_rentals[col] = pd.to_datetime(raw_rentals[col], format='mixed', errors='coerce')

    print(raw_rentals[col].dtype)


datetime64[us]
datetime64[us]


In [14]:
# payment_method — 공백 제거 후 대문자로 통일
raw_rentals['payment_method'] = raw_rentals['payment_method'].str.replace(' ', '').str.upper()

print(raw_rentals['payment_method'].unique())


<StringArray>
['CARD', 'APP', 'MEMBERSHIP']
Length: 3, dtype: str


In [15]:
# 중복 제거 — rental_id 기준
raw_rentals = raw_rentals.drop_duplicates(subset='rental_id', keep='first').reset_index(drop=True)

print(len(raw_rentals))
print(raw_rentals['distance_km'].isna().sum())
print(raw_rentals['fee'].isna().sum())
print(raw_rentals['payment_method'].unique())

14500
200
0
<StringArray>
['CARD', 'APP', 'MEMBERSHIP']
Length: 3, dtype: str


## STEP 4 · 결합

대여 기록에 자전거 정보와 대여소명·자치구를 붙입니다.

```
rentals ─(bike_id)→ bikes ─(station_id)→ stations
```


In [35]:
# how, validate 를 명시할 것
# indicator=True 로 매칭 실패를 확인하고, 몇 건이 왜 실패했는지 설명할 것
# 매칭 실패한 행은 제외하고 진행

bike_merged = (raw_rentals
    .merge(raw_bikes[['bike_id', 'station_id']], how='left', validate='many_to_one', indicator=True, on='bike_id')
    )

print(bike_merged['_merge'].value_counts())
print(bike_merged[bike_merged['_merge'] == 'left_only']['bike_id'].value_counts())

merged = (bike_merged[bike_merged['_merge'] == 'both'].drop(columns=['_merge'])
    .merge(stations, how='left', validate='many_to_one', indicator=True, on='station_id'))

print(merged['_merge'].value_counts())

merged = merged[merged['_merge']=='both'].drop(columns='_merge')

print(stations['district'].isna().sum())

print(len(merged))


_merge
both          14420
left_only        80
right_only        0
Name: count, dtype: int64
bike_id
B099    42
B100    38
Name: count, dtype: int64
_merge
both          14116
left_only       304
right_only        0
Name: count, dtype: int64
0
14116


## STEP 5 · 이상치와 논리 검사

**이 단계가 이 문제의 핵심입니다.**

통계 기법만으로는 찾을 수 없는 오류가 섞여 있습니다. 도메인 규칙으로 걸러내세요.

### 파생 컬럼

먼저 **대여 시간(분)** 을 계산합니다.

```
duration_min = (return_time - rent_time) 의 분 단위
```

### 검사 규칙

| 규칙 | 내용 | 근거 |
| --- | --- | --- |
| ① | 반납 시각이 대여 시각보다 앞서면 안 됨 | 물리적으로 불가능 |
| ② | 요금이 음수면 안 됨 | 논리적으로 불가능 |
| ③ | 이동 거리가 물리 상한을 넘으면 안 됨 | 자전거 속도의 한계 |

> 💡 **물리 상한을 어떻게 구합니까**
자전거로 시속 50km 이상을 지속하는 것은 불가능합니다.
`distance_km / (duration_min / 60)` 이 평균 속도(km/h)이고,
이 값이 **50km/h 를 초과**하면 이상치로 판정하세요.
> 

> ⚠️ **왜 IQR 로는 안 됩니까**
전동 자전거(빠름)와 일반 자전거(느림)가 섞여 있습니다.
일반 자전거에서 비정상적으로 부풀려진 거리값이, 전체 분포에서는 전동의 정상값처럼 보일 수 있습니다.

기대 결과

| 규칙 | 탐지 건수 |
| :--- | :--- |
| ① 반납 <= 대여 | 50 |
| ② 요금 음수 | 20 |
| ③ 속도 > 50km/h | 650 |
| 합계 (중복 제외) | 719 |
| 제거 후 행 수 | 13,701 |


In [ ]:
merged['duration_min'] = (merged['return_time'] - merged['rent_time']).dt.total_seconds() / 60